# 02 — Train gSASRec (Phase 1, Step H)

Обучение per-user скорера на YAMBDA-50m, post-filter (`listen & played_ratio_pct>=50`, `min_pop>=5`).

Запускается в Colab (A100 40GB) или локально на M4 Pro для smoke-теста (`subsample_users`).

**Что делает:**
1. Загружает YAMBDA-50m через `src.data.yambda_loader`.
2. Применяет канонический пайплайн: `filter_listens → filter_min_popularity → build_item_id_to_idx → apply_item_remap → global_temporal_split`.
3. Запускает `train_gsasrec(train, val, TrainConfig(...))` из `src.scorer.train`.
4. Сохраняет `artifacts/gsasrec/best.pt` (checkpoint) + `metrics.csv` + `config.json`.

**Что НЕ делает:** не считает топ-K кэш — это Step I, ноутбук `03_cache_user_scores.ipynb`.

## 0. Setup

In [ ]:
# Colab bootstrap (раскомментировать, если запускаем в Colab):
# !git clone https://github.com/<your-repo>/music-recommendations.git
# %cd music-recommendations
# !pip install -r requirements.txt

In [ ]:
import os, sys
from pathlib import Path


PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print('project root:', PROJECT_ROOT)

import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())

In [ ]:
from src.utils.seed import set_seed

set_seed(42)

## 1. Load + canonical preprocessing

In [ ]:
from src.data.yambda_loader import (
    load_yambda, filter_listens, filter_min_popularity,
    build_item_id_to_idx, apply_item_remap, subsample_users,
)
from src.data.splits import global_temporal_split, SplitConfig


HF_CACHE = os.environ.get('HF_DATASETS_CACHE', None)
raw = load_yambda('50m', cache_dir=HF_CACHE)['interactions']
print('raw events:', len(raw))

In [ ]:
# Smoke-режим (локальный M4 Pro): подсэмплируйте 500-1000 users перед сплитом.
# Полный run на Colab — оставить SMOKE = False.
SMOKE = False
SMOKE_N_USERS = 500

df = filter_listens(raw)
df = filter_min_popularity(df, min_count=5)
if SMOKE:
    df = subsample_users(df, n_users=SMOKE_N_USERS, seed=42)

item_id_to_idx = build_item_id_to_idx(df)
df = apply_item_remap(df, item_id_to_idx)
n_items = len(item_id_to_idx)
print(f'post-filter events: {len(df):,}, users: {df.uid.nunique():,}, items: {n_items:,}')

train, val, test = global_temporal_split(df, SplitConfig())
print(f'train: {len(train):,} ({train.uid.nunique():,} users)')
print(f'val:   {len(val):,} ({val.uid.nunique():,} users)')
print(f'test:  {len(test):,} ({test.uid.nunique():,} users)')

## 2. Train gSASRec

In [ ]:
from src.scorer.train import TrainConfig, train_gsasrec


cfg = TrainConfig(
    n_items=n_items,
    max_seq_len=200,
    hidden_dim=256,
    n_heads=4,
    n_layers=3,
    dropout=0.2,
    n_neg=256,
    gbce_t=0.75,
    batch_size=128,
    eval_batch_size=64,
    lr=1e-3,
    n_epochs=30,
    early_stop_patience=5,
    eval_k=10,
    seed=42,
    out_dir=str(PROJECT_ROOT / 'artifacts' / 'gsasrec'),
    device='cuda' if torch.cuda.is_available() else 'cpu',
    log_every_steps=20,
)
print(cfg)

In [ ]:
result = train_gsasrec(train, val, cfg)
result

## 3. Sanity-check: загрузка чекпоинта

In [ ]:
from src.scorer.inference import load_checkpoint


model, mcfg = load_checkpoint(result['checkpoint'], device=cfg.device)
n_params = sum(p.numel() for p in model.parameters())
print(f'loaded model, {n_params/1e6:.2f}M params, ckpt cfg n_items={mcfg["n_items"]}')

In [ ]:
# Также сохраним item_id_to_idx рядом с чекпоинтом — потребуется в inference и в Phase 2.
from src.utils.caching import save_pickle


save_pickle(item_id_to_idx, PROJECT_ROOT / 'artifacts' / 'gsasrec' / 'item_id_to_idx.pkl')
print('saved item_id_to_idx.pkl')

## 4. Просмотр кривой обучения

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


metrics = pd.read_csv(result['metrics_csv'])
fig, ax1 = plt.subplots(figsize=(7, 4))
ax2 = ax1.twinx()
ax1.plot(metrics['epoch'], metrics['train_loss'], 'b-o', label='train loss')
ax2.plot(metrics['epoch'], metrics[f'val_ndcg@{cfg.eval_k}'], 'r-s', label=f'val NDCG@{cfg.eval_k}')
ax1.set_xlabel('epoch'); ax1.set_ylabel('train loss', color='b'); ax2.set_ylabel(f'val NDCG@{cfg.eval_k}', color='r')
ax1.grid(True, alpha=0.3)
plt.title('gSASRec training')
plt.tight_layout()
plt.show()
metrics.tail()